## Collaborative Filtering
### Patient-patient CF algorithm with MMSE scores at different ages

In [24]:
import numpy as np
import pandas as pd

from numpy import nan

In [ ]:
def patient_cf(df: pd.DataFrame, target: str, metric: str, k: int) -> dict[str, float]:

    # qc checks for program
    if target not in df.index:
        print(f"Error: {target} not found in the dataset.")
        return None

    if sum(df.loc[target].isna()) == 0:
        print(f"Error: {target} has no missing MMSE scores")
        return None
    
    target_idx = df.index.get_loc(target)
    
    # center
    X = df.to_numpy()
    target_preds = np.where(np.isnan(X[target_idx]))[0]
    
    means = np.nanmean(X, axis=1, keepdims=True)
    X = np.where(np.isnan(X), means, X)
    X_centered = X - means

    # target patient
    target_row = X_centered[target_idx]
    X = np.delete(X, target_idx, axis=0)
    
    # calculate similarity
    if metric == "l2":
        sim = -np.linalg.norm(X_centered - target_row, ord=2, axis=1)
    elif metric == "cosine":
        eps = 1e-10
        sim = X_centered.dot(target_row) / (
            (np.linalg.norm(target_row, ord=2) + eps) * (np.linalg.norm(X_centered, ord=2, axis=1) + eps)
        )
    else:
        print(f"Error: invalid metric, please choose one of (l2, cosine)")

    # need to handle zero-variance patients somehow without making medically unsound predictions
    # default to age mean in this case
    if np.all(sim == 0):
        print(f"Zero-variance patient -> defaulting to mean MMSE for each age")
        # code here for alternative prediction
        

    sim = np.delete(sim, target_idx)
    sim_min_max = (sim - sim.min()) / (sim.max() - sim.min())
    sim_patients = np.argpartition(sim_min_max, -k)[-k:]

    # make predictions
    preds = {}
    for target_pred in target_preds:
        pred_item = df.columns[target_pred]
        pred_rating = sum(sim_min_max[sim_patients] * X[sim_patients, target_pred])/sum(sim_min_max[sim_patients])
        preds[pred_item] = pred_rating

    return pd.Series(preds)

In [114]:
mmse_df = pd.read_csv("mmse_user_age_matrix.csv", index_col=0)

# replace missing values for mmse
mmse_df.replace([88, 95, 96, 97, 98, -4], np.float64(nan), inplace=True)
mmse_df.dropna(axis = 0, how="all", inplace=True)
mmse_df.dropna(thresh=5, inplace=True)
mmse_small = mmse_df.head(200)
mmse_small.head()

,18,19,20,21,22,23,24,25,26,27,...,101,102,103,104,105,106,107,108,109,110
NACCID,,,,,,,,,,,,,,,,,,,,,
NACC000162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NACC000225,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NACC000271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NACC000304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NACC000382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# test, all nan for now since we have to deal with zeri-variance
target_patient = mmse_small.index[12]
preds = patient_cf(df=mmse_small, target=target_patient, metric="cosine", k=5)
preds

Zero-variance patient -> defaulting to mean MMSE for each age


C:\Users\annik\AppData\Local\Temp\ipykernel_18876\3683367134.py:43: RuntimeWarning: invalid value encountered in divide
  sim_min_max = (sim - sim.min()) / (sim.max() - sim.min())


18    NaN
19    NaN
20    NaN
21    NaN
22    NaN
       ..
106   NaN
107   NaN
108   NaN
109   NaN
110   NaN
Length: 86, dtype: float64